In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# ---------- 1) metric.csv aggregation ----------
# Ensure timestamp and value types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Aggregations across the full series (global)
agg_count = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(count=('value', 'count'))
agg_ts = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(min_timestamp=('timestamp', 'min'),
                                                           max_timestamp=('timestamp', 'max'))
agg_p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95_value')

metric_summary = pd.concat([agg_count, agg_ts, agg_p95], axis=1).reset_index()

# Convert epoch seconds to timezone-aware datetimes (Asia/Shanghai)
metric_summary['min_timestamp'] = pd.to_datetime(metric_summary['min_timestamp'].astype('float'), unit='s', utc=True).dt.tz_convert(tz)
metric_summary['max_timestamp'] = pd.to_datetime(metric_summary['max_timestamp'].astype('float'), unit='s', utc=True).dt.tz_convert(tz)

# Sort by p95_value descending and take top 20
metric_summary['p95_value'] = metric_summary['p95_value'].astype(float)
metric_summary_top20 = metric_summary.sort_values('p95_value', ascending=False).head(20).reset_index(drop=True)

# Keep columns order as requested
metric_summary_top20 = metric_summary_top20[['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']]

# ---------- 2) trace.csv summary ----------
# Ensure timestamp type
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

trace_unique_trace_names = pd.Series(trace_df['trace_name'].dropna().unique()[:50], name='trace_name')
trace_unique_cmdbs = pd.Series(trace_df['cmdb_id'].dropna().unique()[:50], name='cmdb_id')

if trace_df['timestamp'].notna().any():
    trace_min_ts = pd.to_datetime(trace_df['timestamp'].min(), unit='s', utc=True).tz_convert(tz)
    trace_max_ts = pd.to_datetime(trace_df['timestamp'].max(), unit='s', utc=True).tz_convert(tz)
else:
    trace_min_ts = trace_max_ts = pd.NaT

trace_time_range = pd.Series({'min_timestamp': trace_min_ts, 'max_timestamp': trace_max_ts})

# ---------- 3) log.csv summary ----------
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')

log_unique_log_names = pd.Series(log_df['log_name'].dropna().unique()[:50], name='log_name')
log_unique_cmdbs = pd.Series(log_df['cmdb_id'].dropna().unique()[:50], name='cmdb_id')

if log_df['timestamp'].notna().any():
    log_min_ts = pd.to_datetime(log_df['timestamp'].min(), unit='s', utc=True).tz_convert(tz)
    log_max_ts = pd.to_datetime(log_df['timestamp'].max(), unit='s', utc=True).tz_convert(tz)
else:
    log_min_ts = log_max_ts = pd.NaT

log_time_range = pd.Series({'min_timestamp': log_min_ts, 'max_timestamp': log_max_ts})

# ---------- 4) error_logs.csv summary ----------
error_df['timestamp'] = pd.to_numeric(error_df['timestamp'], errors='coerce').astype('Int64')

error_unique_cmdbs = pd.Series(error_df['cmdb_id'].dropna().unique()[:50], name='cmdb_id')

if error_df['timestamp'].notna().any():
    error_min_ts = pd.to_datetime(error_df['timestamp'].min(), unit='s', utc=True).tz_convert(tz)
    error_max_ts = pd.to_datetime(error_df['timestamp'].max(), unit='s', utc=True).tz_convert(tz)
else:
    error_min_ts = error_max_ts = pd.NaT

error_time_range = pd.Series({'min_timestamp': error_min_ts, 'max_timestamp': error_max_ts})

# Top 20 rows of error_logs.csv with requested columns, convert timestamp to tz-aware datetimes
if not error_df.empty:
    error_top20 = error_df[['timestamp', 'cmdb_id', 'message']].head(20).copy()
    error_top20['timestamp'] = pd.to_datetime(error_top20['timestamp'].astype('float'), unit='s', utc=True).dt.tz_convert(tz)
else:
    error_top20 = error_df[['timestamp', 'cmdb_id', 'message']].head(20)

# Final displayed outputs (compact)
metric_summary_top20, trace_unique_trace_names, trace_unique_cmdbs, trace_time_range, log_unique_log_names, log_unique_cmdbs, log_time_range, error_unique_cmdbs, error_time_range, error_top20

```
Out[1]:
```
```python
# Summarize previous analysis results in plain English as a single compact string.
# Reuse variables from the prior analysis run and produce a short human-readable summary.

top5 = metric_summary_top20.head(5)
total_metric_groups = metric_summary.shape[0] if 'metric_summary' in globals() else metric_summary_top20.shape[0]

trace_names_count = len(trace_unique_trace_names)
trace_cmdbs = trace_unique_cmdbs.tolist()
trace_min_ts = trace_time_range['min_timestamp']
trace_max_ts = trace_time_range['max_timestamp']

log_names = log_unique_log_names.tolist()
log_cmdbs = log_unique_cmdbs.tolist()
log_min_ts = log_time_range['min_timestamp']
log_max_ts = log_time_range['max_timestamp']

error_present = not error_top20.empty
error_cmdbs_list = error_unique_cmdbs.tolist()
error_min_ts = error_time_range['min_timestamp']
error_max_ts = error_time_range['max_timestamp']

lines = []
lines.append(f"Metrics: aggregated into {total_metric_groups} (cmdb_id, kpi_name) groups across the full available series.")
lines.append("Top 5 metric groups by global P95 (cmdb_id | kpi_name | p95_value | count | min_ts → max_ts, all times in UTC+8):")
for i, row in top5.iterrows():
    p95 = f"{row.p95_value:.6g}"
    lines.append(f" {i+1}. {row.cmdb_id} | {row.kpi_name} | p95={p95} | count={int(row.count)} | {row.min_timestamp} → {row.max_timestamp}")

lines.append("")    
lines.append(f"Trace data: {trace_names_count} unique trace_name values listed (showing up to 50). Unique cmdb_id values ({len(trace_cmdbs)}): {', '.join(trace_cmdbs)}.")
lines.append(f" Trace time range: {trace_min_ts} → {trace_max_ts} (UTC+8).")

lines.append("")    
lines.append(f"Log data: unique log_name values: {', '.join(log_names)}.")
lines.append(f" Unique cmdb_id values ({len(log_cmdbs)}): {', '.join(log_cmdbs)}.")
lines.append(f" Log time range: {log_min_ts} → {log_max_ts} (UTC+8).")

lines.append("")
if error_present:
    lines.append(f"Error logs: unique cmdb_id values: {', '.join(error_cmdbs_list)}. Time range: {error_min_ts} → {error_max_ts}. Top 20 error log rows are available for inspection.")
else:
    lines.append("Error logs: no error log records found (error_logs.csv is empty).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp     p95_value
0    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15 2021-03-06 20:30:00+08:00 2021-03-06 20:59:00+08:00  5.169612e+09
1    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     15 2021-03-06 20:30:00+08:00 2021-03-06 20:59:00+08:00  4.674172e+09
2    Redis01  redis-Redis_6379_Redis  (total_commands_proces...     29 2021-03-06 20:31:00+08:00 2021-03-06 20:59:00+08:00  4.545799e+09
3    Redis01                redis-Redis_6379_Redis  (maxmemory)     29 2021-03-06 20:31:00+08:00 2021-03-06 20:59:00+08:00  4.294967e+09
4    Redis02                redis-Redis_6379_Redis  (maxmemory)     29 2021-03-06 20:31:00+08:00 2021-03-06 20:59:00+08:00  4.294967e+09
5   Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
6   Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
7   Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
8   Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
9   Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
10  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
11  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
12  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6 2021-03-06 20:31:00+08:00 2021-03-06 20:56:00+08:00  4.187619e+09
13   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     29 2021-03-06 20:31:00+08:00 2021-03-06 20:59:00+08:00  3.687559e+09
14  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4 2021-03-06 20:38:00+08:00 2021-03-06 20:58:00+08:00  3.455179e+09
15   Redis01            redis-Redis_6379_Redis  (keyspace_hits)     29 2021-03-06 20:31:00+08:00 2021-03-06 20:59:00+08:00  3.440133e+09
16  dockerB1  Container-DOCKER_CONTAINER_b30097144a13--bcou-...      1 2021-03-06 20:42:00+08:00 2021-03-06 20:42:00+08:00  3.072000e+09
17  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...      1 2021-03-06 20:43:00+08:00 2021-03-06 20:43:00+08:00  3.072000e+09
18  dockerA1  Container-DOCKER_CONTAINER_b6760337dc49--bcou-...      1 2021-03-06 20:46:00+08:00 2021-03-06 20:46:00+08:00  3.072000e+09
19  dockerA2  Container-DOCKER_CONTAINER_94eca4f96efe--bcou-...      2 2021-03-06 20:37:00+08:00 2021-03-06 20:49:00+08:00  3.072000e+09, 0            trace.self.duration_mean
1             trace.self.duration_p95
2                trace.self.row_count
3     trace.to_Tomcat01.duration_mean
4      trace.to_Tomcat01.duration_p95
                   ...               
45    trace.to_dockerB1.duration_mean
46     trace.to_dockerB1.duration_p95
47        trace.to_dockerB1.row_count
48    trace.to_dockerB2.duration_mean
49     trace.to_dockerB2.duration_p95
Name: trace_name, Length: 50, dtype: object, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
Name: cmdb_id, dtype: object, min_timestamp   2021-03-06 20:30:00+08:00
max_timestamp   2021-03-06 20:59:00+08:00
dtype: datetime64[ns, Asia/Shanghai], 0     log.error_count
1       log.row_count
2    log.gc_pause_max
Name: log_name, dtype: object, 0     Mysql01
1    Tomcat01
2    Tomcat02
3    Tomcat03
4    Tomcat04
5    apache02
Name: cmdb_id, dtype: object, min_timestamp   2021-03-06 20:30:00+08:00
max_timestamp   2021-03-06 21:00:00+08:00
dtype: datetime64[ns, Asia/Shanghai], Series([], Name: cmdb_id, dtype: object), min_timestamp   NaT
max_timestamp   NaT
dtype: datetime64[ns], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for each (cmdb_id, kpi_name) using full series
threshold_table = (
    metric_df
    .dropna(subset=['cmdb_id', 'kpi_name', 'value'])
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)
threshold_table['p95_value'] = threshold_table['p95_value'].astype(float)

# 2) Filter metric.csv to incident window (inclusive) in UTC+8
start_dt = tz.localize(pd.Timestamp('2021-03-06 20:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-06 21:00:00'))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

window_metrics = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Merge window metrics with thresholds and compute per-group anomaly stats
wm_with_threshold = window_metrics.merge(threshold_table, on=['cmdb_id', 'kpi_name'], how='inner')

# Mark anomalies where value >= p95_value
wm_with_threshold['is_anomaly'] = wm_with_threshold['value'] >= wm_with_threshold['p95_value']

# Aggregations: count_in_window, anomaly_count, max_value_in_window
agg = (
    wm_with_threshold
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        anomaly_count    = ('is_anomaly', 'sum'),
        max_value_in_window = ('value', 'max'),
    )
)

# earliest anomaly timestamp per group (integer seconds)
anomaly_min_ts = (
    wm_with_threshold[wm_with_threshold['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(earliest_anomaly_timestamp=('timestamp', 'min'))
)

# Convert earliest_anomaly_timestamp to tz-aware string representation
if not anomaly_min_ts.empty:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.to_datetime(anomaly_min_ts['earliest_anomaly_timestamp'].astype(float), unit='s', utc=True).dt.tz_convert(tz).astype(str)
else:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.Series(dtype=object)

# Merge earliest anomaly timestamps (may be NaN for groups with zero anomalies)
window_group_summary = agg.merge(anomaly_min_ts, on=['cmdb_id', 'kpi_name'], how='left')

# Attach p95_value by merging threshold_table (should exist)
window_group_summary = window_group_summary.merge(threshold_table, on=['cmdb_id', 'kpi_name'], how='left')

# Compute severity_ratio safely
window_group_summary['severity_ratio'] = window_group_summary['max_value_in_window'] / window_group_summary['p95_value']
window_group_summary.loc[window_group_summary['p95_value'] == 0, 'severity_ratio'] = np.nan

# Ensure column order as requested
window_group_summary = window_group_summary[
    ['cmdb_id', 'kpi_name', 'count_in_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_in_window', 'p95_value', 'severity_ratio']
]

# 4) Sort by anomaly_count desc then severity_ratio desc, limit to top 20 rows
result_top20 = window_group_summary.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Store threshold_table and full per-group window summary in kernel variables for follow-up inspection
# (threshold_table, window_group_summary already available)

# Display compact top-20 result table
result_top20

```
Out[2]:
```
```python
# Build a compact plain-English summary based on the previously computed result_top20.
# Reuse kernel variables: result_top20, threshold_table, window_group_summary

num_rows = len(result_top20)
all_fully_anomalous = (result_top20['anomaly_count'] == result_top20['count_in_window']).all()
earliest_ts_vals = sorted(result_top20['earliest_anomaly_timestamp'].dropna().unique())
earliest_ts_sample = earliest_ts_vals[0] if earliest_ts_vals else "N/A"

# Top severity
max_sev = result_top20['severity_ratio'].max()
top_sev_rows = result_top20[result_top20['severity_ratio'] == max_sev]
top_sev_example = top_sev_rows.iloc[0]
top_example_desc = f"{top_sev_example.cmdb_id} | {top_sev_example.kpi_name} (severity_ratio={top_sev_example.severity_ratio:.3g}, p95={top_sev_example.p95_value:.6g}, max_in_window={top_sev_example.max_value_in_window:.6g})"

lines = [
    f"Summary of window anomaly analysis (top {num_rows} groups):",
    f"- All {num_rows} reported groups had anomalies present in the incident window.",
    f"- For these top groups, anomaly_count equals count_in_window for every group: {all_fully_anomalous}.",
    f"- Earliest anomalies appear at {earliest_ts_sample} (UTC+8) for the shown groups.",
    f"- Highest severity_ratio in the top results is {max_sev:.3g}; an example: {top_example_desc}.",
    "- Notable groups include JVM memory KPIs for IG01/MG01/MG02, various OSLinux process KPIs (IG02, Mysql02, Redis02),",
    "  and several ServiceTest* app.rr / app.sr KPIs — each shows full-window anomalies (31/31 points).",
    "- The per-group global P95 thresholds are stored in variable 'threshold_table'.",
    "- The full per-group window summary (all groups in window with counts, anomaly flags, severity_ratio) is in 'window_group_summary'."
]

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window     p95_value  severity_ratio
0            IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31             31  2021-03-06 20:30:00+08:00         4.000000e-01  3.000000e-01        1.333333
1            IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31  2021-03-06 20:30:00+08:00         2.040136e+09  2.040136e+09        1.000000
2            IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-06 20:30:00+08:00         5.787366e+07  5.787366e+07        1.000000
3            IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31             31  2021-03-06 20:30:00+08:00         3.000000e-01  3.000000e-01        1.000000
4            IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31  2021-03-06 20:30:00+08:00         2.040136e+09  2.040136e+09        1.000000
5            MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31  2021-03-06 20:30:00+08:00         2.040136e+09  2.040136e+09        1.000000
6            MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-06 20:30:00+08:00         5.068759e+08  5.068759e+08        1.000000
7            MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31  2021-03-06 20:30:00+08:00         2.040136e+09  2.040136e+09        1.000000
8            MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31  2021-03-06 20:30:00+08:00         9.473127e+08  9.473127e+08        1.000000
9         Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31  2021-03-06 20:30:00+08:00         1.200000e+01  1.200000e+01        1.000000
10        Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31  2021-03-06 20:30:00+08:00         4.000000e+00  4.000000e+00        1.000000
11   ServiceTest1                                             app.rr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
12   ServiceTest1                                             app.sr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
13  ServiceTest10                                             app.rr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
14  ServiceTest10                                             app.sr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
15  ServiceTest11                                             app.rr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
16  ServiceTest11                                             app.sr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
17   ServiceTest2                                             app.rr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
18   ServiceTest2                                             app.sr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000
19   ServiceTest3                                             app.rr               31             31  2021-03-06 20:30:00+08:00         1.000000e+02  1.000000e+02        1.000000```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 for each (cmdb_id, trace_name) using the full trace series
trace_threshold_table = (
    trace_df
    .dropna(subset=['cmdb_id', 'trace_name', 'value'])
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)
trace_threshold_table['p95_value'] = trace_threshold_table['p95_value'].astype(float)

# 2) Filter trace.csv to the incident window (inclusive) in UTC+8
start_dt = tz.localize(pd.Timestamp('2021-03-06 20:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-06 21:00:00'))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

window_traces = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Restrict to trace_name containing 'duration' (case-insensitive) and allowed cmdb_id list
allowed_cmdbs = ["IG01","IG02","MG01","MG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                 "Mysql01","Mysql02","Redis01","Redis02","apache01","apache02"]

mask_duration = window_traces['trace_name'].str.contains('duration', case=False, na=False)
mask_cmdb = window_traces['cmdb_id'].isin(allowed_cmdbs)
window_traces = window_traces[mask_duration & mask_cmdb].copy()

# 4) Merge with global thresholds (trace_threshold_table) and compute per-group stats
wt_with_thresh = window_traces.merge(trace_threshold_table, on=['cmdb_id', 'trace_name'], how='inner')

# Mark anomalies where value >= p95_value
wt_with_thresh['is_anomaly'] = wt_with_thresh['value'] >= wt_with_thresh['p95_value']

# Aggregations: count_in_window, anomaly_count, max_value_in_window
agg = (
    wt_with_thresh
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        anomaly_count    = ('is_anomaly', 'sum'),
        max_value_in_window = ('value', 'max'),
    )
)

# earliest anomaly timestamp per group
anomaly_min_ts = (
    wt_with_thresh[wt_with_thresh['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(earliest_anomaly_timestamp=('timestamp', 'min'))
)

# Convert earliest_anomaly_timestamp to tz-aware string
if not anomaly_min_ts.empty:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.to_datetime(anomaly_min_ts['earliest_anomaly_timestamp'].astype(float), unit='s', utc=True).dt.tz_convert(tz).astype(str)
else:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.Series(dtype=object)

# Merge aggregations and attach p95_value
trace_window_summary = agg.merge(anomaly_min_ts, on=['cmdb_id', 'trace_name'], how='left')
trace_window_summary = trace_window_summary.merge(trace_threshold_table, on=['cmdb_id', 'trace_name'], how='left')

# Compute severity_ratio
trace_window_summary['severity_ratio'] = trace_window_summary['max_value_in_window'] / trace_window_summary['p95_value']
trace_window_summary.loc[trace_window_summary['p95_value'] == 0, 'severity_ratio'] = np.nan

# Final column ordering
trace_window_summary = trace_window_summary[
    ['cmdb_id', 'trace_name', 'count_in_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_in_window', 'p95_value', 'severity_ratio']
]

# 5) Sort and limit to top 20 rows
result_top20 = trace_window_summary.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep trace_threshold_table and trace_window_summary in kernel variables for follow-up
trace_threshold_table, trace_window_summary, result_top20

```
Out[3]:
```
```python
# Summarize trace analysis results in plain English as a single compact string.
# Reuse kernel variables: trace_threshold_table, trace_window_summary, result_top20

total_threshold_groups = int(trace_threshold_table.shape[0]) if 'trace_threshold_table' in globals() else 0
total_window_groups = int(trace_window_summary.shape[0]) if 'trace_window_summary' in globals() else 0

# How many groups in window had at least one anomaly
if 'trace_window_summary' in globals() and not trace_window_summary.empty:
    groups_with_anomaly = int((trace_window_summary['anomaly_count'] > 0).sum())
    frac_with_anomaly = groups_with_anomaly / total_window_groups if total_window_groups else 0
    # highest severity example
    idx_max = trace_window_summary['severity_ratio'].idxmax()
    top_row = trace_window_summary.loc[idx_max]
    top_desc = (f"{top_row.cmdb_id} | {top_row.trace_name} "
                f"(severity_ratio={top_row.severity_ratio:.3g}, p95={top_row.p95_value:.6g}, "
                f"max_in_window={top_row.max_value_in_window:.6g}, earliest_anomaly={top_row.earliest_anomaly_timestamp})")
    # typical pattern
    common_count_mode = int(trace_window_summary['count_in_window'].mode()[0]) if not trace_window_summary['count_in_window'].empty else None
    common_anomaly_mode = int(trace_window_summary['anomaly_count'].mode()[0]) if not trace_window_summary['anomaly_count'].empty else None
    earliest_anomaly_overall = trace_window_summary['earliest_anomaly_timestamp'].dropna().min() if trace_window_summary['earliest_anomaly_timestamp'].notna().any() else "N/A"
else:
    groups_with_anomaly = 0
    frac_with_anomaly = 0.0
    top_desc = "N/A"
    common_count_mode = None
    common_anomaly_mode = None
    earliest_anomaly_overall = "N/A"

lines = [
    f"Trace analysis summary (duration traces, selected cmdbs):",
    f"- Global P95 thresholds computed for {total_threshold_groups} (cmdb_id, trace_name) groups (variable: trace_threshold_table).",
    f"- {total_window_groups} groups were present in the incident window; {groups_with_anomaly} groups ({frac_with_anomaly:.1%}) had value >= P95 at least once.",
    f"- Most groups in the window show count_in_window={common_count_mode} and anomaly_count={common_anomaly_mode} (mode across groups).",
    f"- Highest observed severity: {top_desc}.",
    f"- Earliest anomaly among reported groups appears around {earliest_anomaly_overall} (UTC+8).",
    f"- Full per-group window summary is stored in 'trace_window_summary' for follow-up inspection.",
    f"- Returned result table was sorted by anomaly_count then severity_ratio and limited to top 20 rows."
]

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                       trace_name   p95_value
0        IG01         trace.self.duration_mean    0.491545
1        IG01          trace.self.duration_p95    2.216800
2        IG01             trace.self.row_count  985.700000
3        IG01  trace.to_Tomcat01.duration_mean    0.397259
4        IG01   trace.to_Tomcat01.duration_p95    2.068725
..        ...                              ...         ...
235  dockerB2       trace.to_MG01.duration_p95    0.354460
236  dockerB2          trace.to_MG01.row_count  146.800000
237  dockerB2      trace.to_MG02.duration_mean    0.060070
238  dockerB2       trace.to_MG02.duration_p95    0.218185
239  dockerB2          trace.to_MG02.row_count  187.200000

[240 rows x 3 columns],       cmdb_id                       trace_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  severity_ratio
0        IG01         trace.self.duration_mean               30              2  2021-03-06 20:46:00+08:00             0.607926   0.491545        1.236765
1        IG01          trace.self.duration_p95               30              2  2021-03-06 20:47:00+08:00             2.332950   2.216800        1.052395
2        IG01  trace.to_Tomcat01.duration_mean               30              2  2021-03-06 20:54:00+08:00             0.407615   0.397259        1.026068
3        IG01   trace.to_Tomcat01.duration_p95               30              2  2021-03-06 20:46:00+08:00             2.127250   2.068725        1.028290
4        IG01  trace.to_Tomcat02.duration_mean               30              2  2021-03-06 20:37:00+08:00             0.562134   0.490967        1.144953
..        ...                              ...              ...            ...                        ...                  ...        ...             ...
111  Tomcat04          trace.self.duration_p95               30              2  2021-03-06 20:47:00+08:00             1.123200   1.097650        1.023277
112  Tomcat04      trace.to_MG01.duration_mean               30              2  2021-03-06 20:37:00+08:00             0.491194   0.411967        1.192313
113  Tomcat04       trace.to_MG01.duration_p95               30              2  2021-03-06 20:37:00+08:00             2.315900   2.145020        1.079664
114  Tomcat04      trace.to_MG02.duration_mean               30              2  2021-03-06 20:37:00+08:00             0.465582   0.368694        1.262788
115  Tomcat04       trace.to_MG02.duration_p95               30              2  2021-03-06 20:37:00+08:00             2.065950   1.831927        1.127747

[116 rows x 8 columns],      cmdb_id                         trace_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  severity_ratio
0   Tomcat03            trace.self.duration_p95               30              2  2021-03-06 20:37:00+08:00             0.927500   0.109710        8.454106
1       MG01   trace.from_dockerB1.duration_p95               30              2  2021-03-06 20:40:00+08:00             0.865200   0.226127        3.826160
2       MG02   trace.from_dockerB2.duration_p95               30              2  2021-03-06 20:41:00+08:00             0.626000   0.218185        2.869125
3       MG01   trace.from_dockerB2.duration_p95               30              2  2021-03-06 20:40:00+08:00             1.012800   0.354460        2.857304
4       MG01            trace.self.duration_p95               30              2  2021-03-06 20:37:00+08:00             1.958550   1.201180        1.630522
5       MG02   trace.from_dockerB1.duration_p95               30              2  2021-03-06 20:30:00+08:00             0.765250   0.472500        1.619577
6       MG02  trace.from_Tomcat03.duration_mean               30              2  2021-03-06 20:38:00+08:00             0.596436   0.372995        1.599047
7   Tomcat03        trace.to_MG02.duration_mean               30              2  2021-03-06 20:38:00+08:00             0.596436   0.372995        1.599047
8       IG02     trace.to_Tomcat02.duration_p95               30              2  2021-03-06 20:54:00+08:00             3.279350   2.171255        1.510348
9   Tomcat02       trace.from_IG02.duration_p95               30              2  2021-03-06 20:54:00+08:00             3.279350   2.171255        1.510348
10      MG01  trace.from_dockerB1.duration_mean               30              2  2021-03-06 20:40:00+08:00             0.082554   0.056378        1.464298
11      MG02  trace.from_dockerA2.duration_mean               30              2  2021-03-06 20:33:00+08:00             0.149857   0.102940        1.455768
12      MG02    trace.to_dockerB1.duration_mean               30              2  2021-03-06 20:38:00+08:00             0.738818   0.517691        1.427141
13      IG02    trace.to_Tomcat03.duration_mean               30              2  2021-03-06 20:38:00+08:00             0.619432   0.445489        1.390455
14  Tomcat03      trace.from_IG02.duration_mean               30              2  2021-03-06 20:38:00+08:00             0.619432   0.445489        1.390455
15      MG01  trace.from_dockerB2.duration_mean               30              2  2021-03-06 20:40:00+08:00             0.085753   0.061835        1.386795
16  Tomcat01            trace.self.duration_p95               30              2  2021-03-06 20:38:00+08:00             0.132700   0.097075        1.366984
17      IG01     trace.to_Tomcat02.duration_p95               30              2  2021-03-06 20:37:00+08:00             2.893950   2.158550        1.340692
18  Tomcat02       trace.from_IG01.duration_p95               30              2  2021-03-06 20:37:00+08:00             2.893950   2.158550        1.340692
19      MG01     trace.to_dockerA2.duration_p95               30              2  2021-03-06 20:37:00+08:00             2.060800   1.561700        1.319588)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import re
import pytz
import numpy as np

# Reuse existing trace_window_summary from kernel
try:
    trace_window_summary
except NameError:
    raise RuntimeError("trace_window_summary not found in kernel. Run previous trace analysis first.")

# Timezone
tz = pytz.timezone('Asia/Shanghai')

# Allowed candidate list
allowed_cmdbs = ["IG01","IG02","MG01","MG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                 "Mysql01","Mysql02","Redis01","Redis02","apache01","apache02"]

# 1) Identify faulty components: unique cmdb_id where anomaly_count > 0 and in allowed list
faulty_mask = (trace_window_summary['anomaly_count'] > 0) & (trace_window_summary['cmdb_id'].isin(allowed_cmdbs))
faulty_components = trace_window_summary.loc[faulty_mask, 'cmdb_id'].dropna().unique()

# 2) Build call graph edges by parsing trace_name patterns across all rows of trace_window_summary
edges = []
to_re = re.compile(r'trace\.to_([A-Za-z0-9_-]+)', flags=re.IGNORECASE)
from_re = re.compile(r'trace\.from_([A-Za-z0-9_-]+)', flags=re.IGNORECASE)

for _, row in trace_window_summary[['cmdb_id', 'trace_name']].dropna().iterrows():
    src = row['cmdb_id']
    name = row['trace_name']
    # find all 'to_X' patterns -> src -> X
    for m in to_re.findall(name):
        edges.append((src, m))
    # find all 'from_Y' patterns -> Y -> src
    for m in from_re.findall(name):
        edges.append((m, src))

call_graph = pd.DataFrame(edges, columns=['source', 'target']).drop_duplicates().reset_index(drop=True)

# Helper to get earliest anomaly timestamp per component (consider only rows with anomaly_count>0)
rows_with_anom = trace_window_summary[trace_window_summary['anomaly_count'] > 0].copy()

# Convert earliest_anomaly_timestamp strings to datetimes (if they are strings), coerce errors
if 'earliest_anomaly_timestamp' in rows_with_anom.columns:
    rows_with_anom['earliest_anomaly_ts_dt'] = pd.to_datetime(rows_with_anom['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
else:
    rows_with_anom['earliest_anomaly_ts_dt'] = pd.NaT

# Compute per-component aggregates for faulty components
records = []
for comp in sorted(faulty_components):
    comp_rows = rows_with_anom[rows_with_anom['cmdb_id'] == comp]
    if comp_rows.empty:
        continue
    # earliest anomaly timestamp (min)
    earliest_dt = comp_rows['earliest_anomaly_ts_dt'].min()
    earliest_str = pd.NaT if pd.isna(earliest_dt) else pd.Timestamp(earliest_dt).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    # max severity_ratio
    max_sev = comp_rows['severity_ratio'].max()
    # faulty_trace_count: number of distinct trace_name rows for that component with anomaly_count>0
    faulty_trace_count = comp_rows['trace_name'].nunique()
    # in_edges: sources where target == comp
    in_edges_list = call_graph.loc[call_graph['target'] == comp, 'source'].dropna().unique().astype(str).tolist()
    out_edges_list = call_graph.loc[call_graph['source'] == comp, 'target'].dropna().unique().astype(str).tolist()
    # Limit to up to 5 unique entries
    in_edges_str = ",".join(in_edges_list[:5])
    out_edges_str = ",".join(out_edges_list[:5])
    records.append({
        'component': comp,
        'earliest_anomaly_timestamp': earliest_str,
        'max_severity_ratio': max_sev,
        'faulty_trace_count': int(faulty_trace_count),
        'in_edges': in_edges_str,
        'out_edges': out_edges_str
    })

faulty_components_df = pd.DataFrame.from_records(records)

# Sort by max_severity_ratio desc then earliest_anomaly_timestamp asc (NaT/empty last)
# Convert earliest_anomaly_timestamp back to datetime for sorting; coerce errors -> NaT
faulty_components_df['earliest_dt_sort'] = pd.to_datetime(faulty_components_df['earliest_anomaly_timestamp'], format='%Y-%m-%d %H:%M:%S%z', utc=True, errors='coerce').dt.tz_convert(tz)
faulty_components_df = faulty_components_df.sort_values(['max_severity_ratio', 'earliest_dt_sort'], ascending=[False, True]).drop(columns=['earliest_dt_sort']).reset_index(drop=True)

# Keep only requested columns and limit to top 20 rows
faulty_components_df = faulty_components_df[['component', 'earliest_anomaly_timestamp', 'max_severity_ratio', 'faulty_trace_count', 'in_edges', 'out_edges']].head(20)

# Store call_graph and faulty_components_df in kernel variables (already assigned)
call_graph, faulty_components_df

```
Out[4]:
```
```python
# Build a compact plain-English summary of the faulty components and call-graph using existing kernel variables.
# Reuse: faulty_components_df, call_graph
num_faulty = len(faulty_components_df)
top_n = min(5, num_faulty)
top_rows = faulty_components_df.sort_values('max_severity_ratio', ascending=False).head(top_n)

lines = []
lines.append(f"Identified {num_faulty} faulty components (anomaly_count>0) within the allowed candidate set.")
lines.append(f"Call-graph contains {len(call_graph)} unique edges (stored in 'call_graph').")
lines.append("")
lines.append(f"Top {top_n} faulty components by max_severity_ratio:")
for _, r in top_rows.iterrows():
    comp = r['component']
    et = r['earliest_anomaly_timestamp'] if pd.notna(r['earliest_anomaly_timestamp']) and r['earliest_anomaly_timestamp']!= 'NaT' else "N/A"
    sev = f"{r['max_severity_ratio']:.3g}" if pd.notna(r['max_severity_ratio']) else "N/A"
    ftc = int(r['faulty_trace_count'])
    ins = r['in_edges'] if r['in_edges'] else "(none)"
    outs = r['out_edges'] if r['out_edges'] else "(none)"
    lines.append(f"- {comp}: earliest_anomaly={et}, max_severity_ratio={sev}, faulty_trace_count={ftc}, in_edges={ins}, out_edges={outs}")

lines.append("")
lines.append("Notes:")
lines.append("- earliest_anomaly_timestamp is the earliest time within the incident window where a trace metric exceeded its global P95.")
lines.append("- in_edges/out_edges show up to 5 unique callers/callees inferred from trace names (pattern 'to_X' or 'from_Y').")
lines.append("- Full per-component table is available in variable 'faulty_components_df'; full call edges are in 'call_graph'.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   source    target
0    IG01  Tomcat01
1    IG01  Tomcat02
2    IG01  Tomcat03
3    IG01  Tomcat04
4    IG02  Tomcat01
..    ...       ...
29   MG02  dockerA2
30   MG02  dockerB1
31   MG02  dockerB2
32   root  Tomcat01
33   root  Tomcat02

[34 rows x 2 columns],   component earliest_anomaly_timestamp  max_severity_ratio  faulty_trace_count                                      in_edges                            out_edges
0  Tomcat03   2021-03-06 20:37:00+0800            8.454106                  10                                     IG01,IG02                            MG01,MG02
1      MG01   2021-03-06 20:30:00+0800            3.826160                  26  Tomcat01,Tomcat02,Tomcat03,Tomcat04,dockerA1  dockerA1,dockerA2,dockerB1,dockerB2
2      MG02   2021-03-06 20:30:00+0800            2.869125                  26  Tomcat01,Tomcat02,Tomcat03,Tomcat04,dockerA1  dockerA1,dockerA2,dockerB1,dockerB2
3  Tomcat02   2021-03-06 20:30:00+0800            1.510348                  12                                IG01,IG02,root                            MG01,MG02
4      IG02   2021-03-06 20:37:00+0800            1.510348                  10                                                Tomcat01,Tomcat02,Tomcat03,Tomcat04
5  Tomcat01   2021-03-06 20:30:00+0800            1.366984                  12                                IG01,IG02,root                            MG01,MG02
6      IG01   2021-03-06 20:37:00+0800            1.340692                  10                                                Tomcat01,Tomcat02,Tomcat03,Tomcat04
7  Tomcat04   2021-03-06 20:37:00+0800            1.292513                  10                                     IG01,IG02                            MG01,MG02)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Target components
mg_ids = ["MG01", "MG02"]

# 1) Compute global P95 for each (cmdb_id, kpi_name) for MG01 and MG02 using full series
mg_threshold_table = (
    metric_df[metric_df['cmdb_id'].isin(mg_ids)]
    .dropna(subset=['cmdb_id', 'kpi_name', 'value'])
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_value'})
)
mg_threshold_table['p95_value'] = mg_threshold_table['p95_value'].astype(float)

# 2) Filter metric.csv to the incident window (inclusive) and keep only MG01/MG02 rows
start_dt = tz.localize(pd.Timestamp('2021-03-06 20:30:00'))
end_dt   = tz.localize(pd.Timestamp('2021-03-06 21:00:00'))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

mg_window = metric_df[
    (metric_df['cmdb_id'].isin(mg_ids)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# 3) For each (cmdb_id, kpi_name) in the window compute stats using the global p95 thresholds
# Merge window data with thresholds (inner join ensures we skip groups without global p95)
mg_merged = mg_window.merge(mg_threshold_table, on=['cmdb_id', 'kpi_name'], how='inner')

# Mark anomalies
mg_merged['is_anomaly'] = mg_merged['value'] >= mg_merged['p95_value']

# Aggregations
agg = (
    mg_merged
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_in_window = ('value', 'count'),
        anomaly_count    = ('is_anomaly', 'sum'),
        max_value_in_window = ('value', 'max'),
    )
)

# earliest anomaly timestamp per group (if any)
anomaly_min_ts = (
    mg_merged[mg_merged['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(earliest_anomaly_timestamp=('timestamp', 'min'))
)

# Convert earliest anomaly ts to tz-aware string
if not anomaly_min_ts.empty:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.to_datetime(
        anomaly_min_ts['earliest_anomaly_timestamp'].astype(float), unit='s', utc=True
    ).dt.tz_convert(tz).astype(str)
else:
    anomaly_min_ts['earliest_anomaly_timestamp'] = pd.Series(dtype=object)

# Merge aggregations and attach p95_value
mg_window_summary = agg.merge(anomaly_min_ts, on=['cmdb_id', 'kpi_name'], how='left')
mg_window_summary = mg_window_summary.merge(mg_threshold_table, on=['cmdb_id', 'kpi_name'], how='left')

# Compute severity_ratio
mg_window_summary['severity_ratio'] = mg_window_summary['max_value_in_window'] / mg_window_summary['p95_value']
mg_window_summary.loc[mg_window_summary['p95_value'] == 0, 'severity_ratio'] = np.nan

# 4) Final result table sorted by severity_ratio desc then anomaly_count desc, top 20
result_table = mg_window_summary[
    ['cmdb_id', 'kpi_name', 'count_in_window', 'anomaly_count', 'earliest_anomaly_timestamp',
     'max_value_in_window', 'p95_value', 'severity_ratio']
].copy()

result_top20 = result_table.sort_values(['severity_ratio', 'anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Also prepare a short list (up to 10) of distinct kpi_name patterns for MG01 and MG02
mg_kpi_patterns = pd.Series(
    mg_threshold_table['kpi_name'].dropna().unique()[:10],
    name='kpi_name'
)

# Store variables in kernel: mg_threshold_table, mg_window_summary (already assigned)
mg_threshold_table, mg_window_summary, result_top20, mg_kpi_patterns

```
Out[5]:
```
```python
# Summarize MG01/MG02 metric anomaly analysis in plain English (compact).
# Reuse kernel variables: mg_threshold_table, mg_window_summary, result_top20, mg_kpi_patterns

total_thresholds = int(mg_threshold_table.shape[0]) if 'mg_threshold_table' in globals() else 0
total_window_groups = int(mg_window_summary.shape[0]) if 'mg_window_summary' in globals() else 0
groups_with_anom = int((mg_window_summary['anomaly_count'] > 0).sum()) if 'mg_window_summary' in globals() else 0

# Prepare top anomalous KPIs summary (up to 10)
top_n = result_top20.head(10) if 'result_top20' in globals() else None
top_lines = []
if top_n is not None and not top_n.empty:
    for _, r in top_n.iterrows():
        sev = (f"{r.severity_ratio:.3g}" if pd.notna(r.severity_ratio) else "NaN")
        ea = r.earliest_anomaly_timestamp if pd.notna(r.earliest_anomaly_timestamp) else "N/A"
        top_lines.append(f"{r.cmdb_id} | {r.kpi_name} | severity_ratio={sev} | anomaly_count={int(r.anomaly_count)} | earliest={ea}")

kpi_sample = mg_kpi_patterns.tolist() if 'mg_kpi_patterns' in globals() else []

summary_lines = [
    f"MG01/MG02 metric anomaly summary (incident window 2021-03-06 20:30→21:00 UTC+8):",
    f"- Global P95 thresholds computed for {total_thresholds} (cmdb_id, kpi_name) groups (variable: mg_threshold_table).",
    f"- {total_window_groups} metric groups were present in the incident window for MG01/MG02; {groups_with_anom} groups had values >= their global P95 at least once.",
    f"- Top anomalous resource KPIs (up to 10):"
]
summary_lines += [f"  - {line}" for line in top_lines]
summary_lines += [
    "- Key observations:",
    "  - Several JVM memory KPIs for MG01 show full-window anomalies (HeapMemoryMax/NoHeapMemoryUsed), indicating memory-related signals.",
    "  - Disk I/O and per-disk KPIs (DSKRead/DSKTps/DSKBps/DSKRTps) for both MG01 and MG02 show elevated severity ratios (typically 1.3–1.8× P95).",
    "  - Some TCP and CPU wait metrics (e.g., TCP-CLOSE-WAIT, CPUWio, CPULoad/SingleCpuUtil) also exceed P95, with TCP-CLOSE-WAIT in MG01 showing the highest severity_ratio (~20×).",
    "  - Severity ratios and earliest anomaly timestamps are included per-row above; earliest anomalies appear mostly between 20:30 and 20:54 (UTC+8).",
    f"- Example KPI name patterns (sample up to 10) to inspect further: {', '.join(kpi_sample[:10])}.",
    "- Detailed tables are stored in kernel variables: mg_threshold_table (global P95s) and mg_window_summary (per-group window stats)."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                          kpi_name     p95_value
0      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
1      MG01        JVM-Memory_7779_JVM_Memory_HeapMemoryUsage  5.112410e+01
2      MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.049445e+09
3      MG01       JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
4      MG01         JVM-Operating System_7779_JVM_JVM_CPULoad  1.110400e+00
..      ...                                               ...           ...
179    MG02                      OSLinux-OSLinux_SWAP_SWAP_Si  0.000000e+00
180    MG02                      OSLinux-OSLinux_SWAP_SWAP_So  0.000000e+00
181    MG02  OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute  1.000000e+00
182    MG02      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname  1.000000e+00
183    MG02                OSLinux-OSLinux_ZABBIX_Host_Uptime  6.234627e+06

[184 rows x 3 columns],     cmdb_id                                          kpi_name  count_in_window  anomaly_count  max_value_in_window earliest_anomaly_timestamp     p95_value  severity_ratio
0      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31         2.040136e+09  2021-03-06 20:30:00+08:00  2.040136e+09        1.000000
1      MG01        JVM-Memory_7779_JVM_Memory_HeapMemoryUsage               20              1         5.214820e+01  2021-03-06 20:54:00+08:00  5.112410e+01        1.020032
2      MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryUsed               31              2         1.063895e+09  2021-03-06 20:53:00+08:00  1.049445e+09        1.013769
3      MG01       JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31         5.068759e+08  2021-03-06 20:30:00+08:00  5.068759e+08        1.000000
4      MG01         JVM-Operating System_7779_JVM_JVM_CPULoad               21              2         1.157500e+00  2021-03-06 20:47:00+08:00  1.110400e+00        1.042417
..      ...                                               ...              ...            ...                  ...                        ...           ...             ...
179    MG02                      OSLinux-OSLinux_SWAP_SWAP_Si                8              8         0.000000e+00  2021-03-06 20:32:00+08:00  0.000000e+00             NaN
180    MG02                      OSLinux-OSLinux_SWAP_SWAP_So                8              8         0.000000e+00  2021-03-06 20:32:00+08:00  0.000000e+00             NaN
181    MG02  OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute                2              2         1.000000e+00  2021-03-06 20:40:00+08:00  1.000000e+00        1.000000
182    MG02      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname                2              2         1.000000e+00  2021-03-06 20:47:00+08:00  1.000000e+00        1.000000
183    MG02                OSLinux-OSLinux_ZABBIX_Host_Uptime               19              1         6.234735e+06  2021-03-06 21:00:00+08:00  6.234627e+06        1.000017

[184 rows x 8 columns],    cmdb_id                                           kpi_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  p95_value  severity_ratio
0     MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               20              1  2021-03-06 20:32:00+08:00               1.0000   0.050000       20.000000
1     MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps               10              1  2021-03-06 20:41:00+08:00               1.0000   0.550000        1.818182
2     MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               10              1  2021-03-06 20:56:00+08:00              15.0000   8.250000        1.818182
3     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                9              1  2021-03-06 20:40:00+08:00               0.2000   0.120000        1.666667
4     MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                9              1  2021-03-06 20:40:00+08:00               6.0000   3.600000        1.666667
5     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                9              1  2021-03-06 20:40:00+08:00               0.1833   0.109980        1.666667
6     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                9              1  2021-03-06 20:40:00+08:00               6.0000   3.600000        1.666667
7     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                9              1  2021-03-06 20:59:00+08:00               0.2000   0.120000        1.666667
8     MG02                             OSLinux-CPU_CPU_CPUWio               18              1  2021-03-06 20:49:00+08:00               0.5198   0.325490        1.596977
9     MG01                             OSLinux-CPU_CPU_CPUWio               20              1  2021-03-06 20:50:00+08:00               0.2994   0.208580        1.435420
10    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                9              1  2021-03-06 20:40:00+08:00               1.3000   0.980000        1.326531
11    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps               10              1  2021-03-06 20:55:00+08:00               2.0000   1.550000        1.290323
12    MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps               10              1  2021-03-06 20:34:00+08:00               7.0000   6.100000        1.147541
13    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                9              1  2021-03-06 20:59:00+08:00               0.4000   0.360000        1.111111
14    MG02                            OSLinux-CPU_CPU_CPULoad               18              1  2021-03-06 20:51:00+08:00               1.2900   1.162500        1.109677
15    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                9              1  2021-03-06 20:41:00+08:00              17.0000  15.400000        1.103896
16    MG02                    OSLinux-CPU_CPU-1_SingleCpuUtil               11              1  2021-03-06 20:57:00+08:00               3.9507   3.590650        1.100274
17    MG02         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage               22              2  2021-03-06 20:40:00+08:00              58.5483  53.263870        1.099212
18    MG01                    OSLinux-CPU_CPU-3_SingleCpuUtil               10              1  2021-03-06 21:00:00+08:00               3.2280   2.981715        1.082598
19    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps                9              1  2021-03-06 20:40:00+08:00               0.9833   0.916660        1.072699, 0       JVM-Memory_7779_JVM_Memory_HeapMemoryMax
1     JVM-Memory_7779_JVM_Memory_HeapMemoryUsage
2      JVM-Memory_7779_JVM_Memory_HeapMemoryUsed
3    JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed
4      JVM-Operating System_7779_JVM_JVM_CPULoad
5                JVM-Runtime_7779_JVM_JVM_Uptime
6       JVM-Threads_7779_JVM_ThreadCount_Threads
7                OSLinux-CPU_CPU-0_SingleCpuUtil
8                OSLinux-CPU_CPU-0_SingleCpuidle
9                OSLinux-CPU_CPU-1_SingleCpuUtil
Name: kpi_name, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df and mg_threshold_table if present; otherwise load/compute
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure numeric types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    mg_threshold_table
except NameError:
    # compute global P95 for MG01/MG02 if not present
    mg_threshold_table = (
        metric_df[metric_df['cmdb_id'].isin(['MG01', 'MG02'])]
        .dropna(subset=['cmdb_id', 'kpi_name', 'value'])
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95_value'})
    )
mg_threshold_table['p95_value'] = mg_threshold_table['p95_value'].astype(float)

# Timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(pd.Timestamp('2021-03-06 20:30:00')).timestamp())
end_ts   = int(tz.localize(pd.Timestamp('2021-03-06 21:00:00')).timestamp())

# KPI list (exact matches)
kpis = [
    "OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps",
    "OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps",
    "JVM-Memory_7779_JVM_Memory_HeapMemoryMax",
    "JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed",
    "OSLinux-CPU_CPU_CPUWio"
]

# 1) Retrieve all metric rows in the incident window for MG01 and the given KPIs
mg01_raw = metric_df[
    (metric_df['cmdb_id'] == 'MG01') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['kpi_name'].isin(kpis))
].copy()

# Convert timestamp to tz-aware datetime for display, sort ascending
if not mg01_raw.empty:
    mg01_raw['timestamp'] = pd.to_datetime(mg01_raw['timestamp'].astype(float), unit='s', utc=True).dt.tz_convert(tz)
    mg01_raw = mg01_raw[['timestamp', 'cmdb_id', 'kpi_name', 'value']].sort_values('timestamp', ascending=True).reset_index(drop=True)
else:
    mg01_raw = mg01_raw[['timestamp', 'cmdb_id', 'kpi_name', 'value']]

# Limit to 20 rows for compact display
mg01_raw_rows = mg01_raw.head(20).copy()

# 2) Per-KPI 1-row aggregates using global p95s from mg_threshold_table
records = []
for k in kpis:
    # find p95 for MG01,k
    row = mg_threshold_table[(mg_threshold_table['cmdb_id'] == 'MG01') & (mg_threshold_table['kpi_name'] == k)]
    if row.empty:
        p95_value = np.nan
    else:
        p95_value = float(row['p95_value'].iloc[0])
    # window rows for this KPI
    grp = mg01_raw[mg01_raw['kpi_name'] == k].copy()
    if grp.empty:
        earliest_breach = None
        max_val = np.nan
    else:
        max_val = grp['value'].max()
        # earliest breach where value >= p95_value (only if p95 is not nan)
        if not np.isnan(p95_value):
            breaches = grp[grp['value'] >= p95_value]
            if not breaches.empty:
                earliest_breach_ts = breaches['timestamp'].min()
                earliest_breach = pd.Timestamp(earliest_breach_ts).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
            else:
                earliest_breach = None
        else:
            earliest_breach = None
    # severity ratio
    if pd.isna(p95_value) or p95_value == 0:
        severity_ratio = np.nan
    else:
        # if max_val is nan then severity is nan
        severity_ratio = (max_val / p95_value) if not pd.isna(max_val) else np.nan

    records.append({
        'kpi_name': k,
        'p95_value': p95_value,
        'earliest_breach_timestamp': earliest_breach,
        'max_value_in_window': max_val,
        'severity_ratio': severity_ratio
    })

mg01_kpi_aggregates = pd.DataFrame.from_records(records, columns=['kpi_name', 'p95_value', 'earliest_breach_timestamp', 'max_value_in_window', 'severity_ratio'])

# Keep outputs compact: mg01_raw_rows (<=20 rows) and mg01_kpi_aggregates (one row per KPI)
mg01_raw_rows, mg01_kpi_aggregates

```
Out[6]:
```
Summary for MG01 (incident window 2021-03-06 20:30 → 21:00 UTC+8):

Raw rows returned (sample, sorted by timestamp; stored in variable mg01_raw_rows): 20 rows of metric points for the requested KPIs (timestamp, cmdb_id, kpi_name, value). These include repeated JVM memory metrics at 20:30+, CPUWio values, disk and TCP metrics (see stored mg01_raw_rows for full rows).

Per-KPI aggregates (one row per KPI; stored in variable mg01_kpi_aggregates):
- OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
  - p95 = 0.05
  - earliest breach = 2021-03-06 20:32:00+08:00
  - max in window = 1.0
  - severity_ratio = 20.0 (very large spike)
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead
  - p95 = 3.6
  - earliest breach = 2021-03-06 20:40:00+08:00
  - max in window = 6.0
  - severity_ratio ≈ 1.667
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps
  - p95 = 0.55
  - earliest breach = 2021-03-06 20:41:00+08:00
  - max in window = 1.0
  - severity_ratio ≈ 1.818
- OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps
  - p95 = 6.1
  - earliest breach = 2021-03-06 20:34:00+08:00
  - max in window = 7.0
  - severity_ratio ≈ 1.148
- JVM-Memory_7779_JVM_Memory_HeapMemoryMax
  - p95 = 2.040136e+09
  - earliest breach = 2021-03-06 20:30:00+08:00
  - max in window = 2.040136e+09
  - severity_ratio = 1.0 (at threshold)
- JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed
  - p95 = 5.068759e+08
  - earliest breach = 2021-03-06 20:30:00+08:00
  - max in window = 5.068759e+08
  - severity_ratio = 1.0 (at threshold)
- OSLinux-CPU_CPU_CPUWio
  - p95 = 0.20858
  - earliest breach = 2021-03-06 20:50:00+08:00
  - max in window = 0.2994
  - severity_ratio ≈ 1.435

Key takeaways:
- The most extreme anomaly is TCP-CLOSE-WAIT (severity_ratio 20×) at ~20:32, indicating many TCP connections in CLOSE-WAIT state.
- Disk I/O KPIs (DSKRead, DSKTps, DSKBps) show moderate breaches (severity 1.15–1.82×).
- JVM memory KPIs are at or equal to their P95 (severity 1.0) starting at 20:30.
- CPUWio shows a moderate breach later (~20:50).

Stored variables for follow-up:
- mg01_raw_rows — the returned raw metric rows (limited to 20 shown).
- mg01_kpi_aggregates — one-row aggregates per requested KPI (p95, earliest breach, max, severity_ratio).

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id                                         kpi_name         value
0  2021-03-06 20:30:00+08:00    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
1  2021-03-06 20:30:00+08:00    MG01      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
2  2021-03-06 20:30:00+08:00    MG01                           OSLinux-CPU_CPU_CPUWio  8.730000e-02
3  2021-03-06 20:30:00+08:00    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  0.000000e+00
4  2021-03-06 20:30:00+08:00    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00
5  2021-03-06 20:31:00+08:00    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
6  2021-03-06 20:31:00+08:00    MG01      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
7  2021-03-06 20:31:00+08:00    MG01                           OSLinux-CPU_CPU_CPUWio  3.740000e-02
8  2021-03-06 20:32:00+08:00    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
9  2021-03-06 20:32:00+08:00    MG01      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
10 2021-03-06 20:32:00+08:00    MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.000000e+00
11 2021-03-06 20:32:00+08:00    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1.000000e+00
12 2021-03-06 20:33:00+08:00    MG01   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00
13 2021-03-06 20:33:00+08:00    MG01                           OSLinux-CPU_CPU_CPUWio  2.490000e-02
14 2021-03-06 20:33:00+08:00    MG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  0.000000e+00
15 2021-03-06 20:33:00+08:00    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
16 2021-03-06 20:33:00+08:00    MG01      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
17 2021-03-06 20:34:00+08:00    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
18 2021-03-06 20:34:00+08:00    MG01      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08
19 2021-03-06 20:34:00+08:00    MG01                           OSLinux-CPU_CPU_CPUWio  4.160000e-02,                                           kpi_name     p95_value earliest_breach_timestamp  max_value_in_window  severity_ratio
0   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  5.000000e-02  2021-03-06 20:32:00+0800         1.000000e+00       20.000000
1  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  3.600000e+00  2021-03-06 20:40:00+0800         6.000000e+00        1.666667
2   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  5.500000e-01  2021-03-06 20:41:00+0800         1.000000e+00        1.818182
3   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  6.100000e+00  2021-03-06 20:34:00+0800         7.000000e+00        1.147541
4         JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2021-03-06 20:30:00+0800         2.040136e+09        1.000000
5      JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.068759e+08  2021-03-06 20:30:00+0800         5.068759e+08        1.000000
6                           OSLinux-CPU_CPU_CPUWio  2.085800e-01  2021-03-06 20:50:00+0800         2.994000e-01        1.435420)```
```